# SCARF on Google Colab for SCP3357

This notebook clones `HopTF`, bootstraps a Colab-compatible SCARF environment, prepares an SCP3357 AnnData input, computes `obsm['X_scarf']` on GPU, and uploads the finished `.h5ad` to Hugging Face.

Notes:
- Use a **GPU** Colab runtime.
- The SCARF bootstrap step may download the official `model_files.zip` bundle from Zenodo, so expect a multi-gigabyte transfer the first time.
- If the legacy SCARF median pickle is unavailable from the public bundles, the embedding script will derive approximate per-gene nonzero medians from the input dataset instead.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/cpudan/HopTF.git"
REPO_REF = "dan"
WORKDIR = Path("/content/HopTF")

# Option A: build from an SCP3357 portal bundle already present in the runtime.
SCP3357_INPUT_DIR = None  # Example: "/content/SCP3357"

# Option B: skip the build step and start from an existing AnnData file.
INPUT_H5AD = None  # Example: "/content/my_existing_input.h5ad"

RAW_H5AD = WORKDIR / "data/processed/scp3357/SCP3357_raw_counts.h5ad"
OUTPUT_H5AD = WORKDIR / "data/processed/scp3357/SCP3357_raw_counts_scarf.h5ad"

UPLOAD_TO_HF = True
HF_REPO_ID = "pvd232/HopTF"
HF_REPO_TYPE = "dataset"
HF_PATH_IN_REPO = f"scp3357/{OUTPUT_H5AD.name}"


In [ ]:
from datetime import datetime
import os
import shutil
import subprocess


def log_step(message: str) -> None:
    timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
    print(f"[{timestamp}] {message}", flush=True)


def run_cmd(command, cwd=None) -> None:
    if cwd is None:
        cwd = os.getcwd()
    log_step(f"Running command in {cwd}: {' '.join(command)}")
    subprocess.run(command, cwd=cwd, check=True)
    log_step(f"Finished command: {' '.join(command)}")


In [ ]:
if WORKDIR.exists():
    log_step(f"Removing existing checkout at {WORKDIR}")
    shutil.rmtree(WORKDIR)

run_cmd([
    "git",
    "clone",
    "--depth",
    "1",
    "--branch",
    REPO_REF,
    REPO_URL,
    str(WORKDIR),
])
os.chdir(WORKDIR)
log_step(f"Working directory is now {os.getcwd()}")


In [ ]:
run_cmd(["bash", "scripts/bootstrap_scarf_colab.sh"], cwd=str(WORKDIR))


## Prepare the SCP3357 AnnData input

Choose one of the following:
- Set `INPUT_H5AD` above if you already have a `.h5ad` with `layers['counts']`.
- Set `SCP3357_INPUT_DIR` above if you want to build the `.h5ad` from a Single Cell Portal bundle inside Colab.


In [ ]:
if INPUT_H5AD is None:
    if SCP3357_INPUT_DIR is None:
        raise ValueError("Set either INPUT_H5AD or SCP3357_INPUT_DIR before running this cell.")
    run_cmd([
        "python",
        "scripts/build_scp3357_h5ad.py",
        "--input-dir",
        str(SCP3357_INPUT_DIR),
        "--output",
        str(RAW_H5AD),
    ], cwd=str(WORKDIR))
    input_h5ad = RAW_H5AD
else:
    input_h5ad = Path(INPUT_H5AD)
    if not input_h5ad.exists():
        raise FileNotFoundError(input_h5ad)

log_step(f"Input AnnData: {input_h5ad}")


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Switch Colab to a GPU runtime before computing SCARF embeddings.")

run_cmd([
    "python",
    "scripts/add_scarf_embeddings.py",
    "--input-h5ad",
    str(input_h5ad),
    "--output-h5ad",
    str(OUTPUT_H5AD),
    "--device",
    "cuda",
], cwd=str(WORKDIR))


In [ ]:
import anndata as ad

log_step(f"Inspecting output file {OUTPUT_H5AD}")
adata = ad.read_h5ad(OUTPUT_H5AD, backed="r")
print("shape:", adata.shape)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("X_scarf shape:", adata.obsm["X_scarf"].shape)
print("median source:", adata.uns.get("scarf_embedding", {}).get("median_source"))
adata.file.close()


In [ ]:
if UPLOAD_TO_HF:
    from huggingface_hub import HfApi, notebook_login

    log_step("Authenticating with Hugging Face")
    notebook_login()

    log_step(f"Uploading {OUTPUT_H5AD} to hf://datasets/{HF_REPO_ID}/{HF_PATH_IN_REPO}")
    api = HfApi()
    api.upload_file(
        path_or_fileobj=str(OUTPUT_H5AD),
        path_in_repo=HF_PATH_IN_REPO,
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        commit_message=f"Add SCP3357 SCARF embeddings: {OUTPUT_H5AD.name}",
    )
    log_step("Hugging Face upload finished")
else:
    log_step("Skipping Hugging Face upload")
